In [133]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [166]:
X, y = load_breast_cancer(return_X_y= True)


In [198]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scale = StandardScaler()
x_train_scaled = scale.fit_transform(x_train)
x_test_scaled = scale.transform(x_test)

type(x_train_scaled), x_train_scaled.shape, y.shape


(numpy.ndarray, (455, 30), (569,))

## Hyperparameters

In [248]:
m = len(x_train_scaled)
lr = .1

ReLU = lambda x: np.maximum(x, 0)
Sigmoid = lambda x: 1/(1 + np.exp(-x))
def deriv_ReLU(x):
    x = x.copy()
    x[np.where(x> 0)] = 1
    x[np.where(x<= 0)] = 0
    return x

<img src='Images\notebook\NN.DeapLearning.AI.jpeg' width=400/>

In [261]:
shapes = [(3, X.shape[1]), (5, 3), (4, 5), (2, 4), (1, 2)]

W = []
B = []
for shape in shapes:
    w = np.random.randn(*shape) * np.sqrt(2.0 / shape[1])
    b = np.random.random((len(w), 1))
    W.append(w)
    B.append(b)


<img src='Images\notebook\Forw_Backw.DeapLearning.AI.jpeg' width=700/>

In [262]:
for i in range(700):
    
    # Caches
    cache_a = []; cache_z = []; cache_w = []; cache_b = []   
    
# Forward propagation
    # Input layer
    a = x_train_scaled.T             #(n, m)
    cache_a.append(a)
    
    # Hidden layers
    for w,b in zip(W[:-1], B[:-1]):  # last layer has sigmoid activation
        z = w @ a + b                # (nodes, n) @ (n, m) + (nodes, 1)
        a = ReLU(z)
        cache_a.append(a); cache_z.append(z); cache_w.append(w); cache_b.append(b)
    # cache_a.pop()
    # Output layer
    w, b = W[-1], B[-1]
    z = w @ a + b              
    y_hat = Sigmoid(z)
    
# Backward propagation
    # Output layer
    dz = y_hat - y_train      # (1, m)
    dw = (1/m) * (dz @ a.T)     # (1, m) (m, 2)
    db = (1/m) * np.sum(dz, axis=1, keepdims= True)
    
    w = w - lr * dw  
    b = b - lr * db 
    W[-1], B[-1] = w, b
    
    
    # Hidden layers
    for i, (a, z) in enumerate(zip(cache_a[::-1][1:], cache_z[::-1]), 1):
        
        w, b = W[-i], B[-i]
        
        dz = (w.T @ dz) * deriv_ReLU(z) # (1, 1) (1,m) (2, m) = (2, m)
        dw = (1/m) * (dz @ a.T)     # (2, m) (m, 2)
        db = (1/m) * np.sum(dz, axis=1, keepdims= True)  
        
        W[-(i+1)] -=  lr * dw  
        B[-(i + 1)] -= lr * db  
        
        # print(i)
    
        
    # for w in W:
    #     print(w.shape)
    
    # break
        
        
        
    
y_hat.shape

(1, 455)

<img src='Images\notebook\Forw_Backw.DeapLearning.AI.jpeg' width=700/>

In [275]:
# Input layer
a = x_test_scaled.T         

# Hidden layers
for w,b in zip(W[:-1], B[:-1]):  
    z = w @ a + b               
    a = ReLU(z)


# Output layer
w, b = W[-1], B[-1]
z = w @ a + b              
y_hat = Sigmoid(z)

In [276]:
y_hat = np.round(y_hat).astype("int")

In [277]:
from sklearn.metrics import accuracy_score

In [279]:
y_hat.shape, y_test.shape

((1, 114), (114,))

In [281]:
accuracy_score(y_hat.flatten(), y_test)

0.9736842105263158